# Spike 02: model2vec-rs

**Goal:** Validate that the `model2vec-rs` crate (official MinishLab) can:
1. Download and load a POTION model from HuggingFace
2. Embed text and return `Vec<f32>`
3. Embed a batch of texts
4. Report the output dimension
5. Resolve the model revision (Git commit SHA)

**API** (from docs.rs):
- Struct: `StaticModel`
- Load: `StaticModel::from_pretrained(repo, token: Option, normalize: Option<bool>, subfolder: Option<&str>)`
- Single: `model.encode_single(sentence)` → `Vec<f32>`
- Batch: `model.encode(sentences)` → `Vec<Vec<f32>>`

In [3]:
:dep model2vec-rs = "*"

## 1. Load a model

Try the default: `minishlab/potion-base-2M` (smallest, fastest to download for testing)

In [4]:
use model2vec_rs::model::StaticModel;

let model = StaticModel::from_pretrained("minishlab/potion-base-2M", None, None, None).unwrap();
println!("Model loaded successfully");

Model loaded successfully


## 2. Embed a single string

In [5]:
let embedding = model.encode_single("Rust ownership and borrowing");

println!("Dimension: {}", embedding.len());
println!("First 10 values: {:?}", &embedding[..10]);

Dimension: 64
First 10 values: [0.12788704, 0.21786925, 0.16368529, -0.033601012, 0.03558616, -0.01817493, -0.0071195913, -0.016380342, -0.11628809, -0.013932007]


## 3. Batch embedding

In [7]:
let texts = vec![
    "CRDT conflict resolution in distributed systems".to_string(),
    "Rust ownership model prevents data races".to_string(),
    "Making sourdough bread with a Dutch oven".to_string(),
];

let batch_embeddings = model.encode(&texts);

println!("Batch size: {}", batch_embeddings.len());
for (i, emb) in batch_embeddings.iter().enumerate() {
    println!("  [{}] dim={}, first 5: {:?}", i, emb.len(), &emb[..5]);
}

Batch size: 3
  [0] dim=64, first 5: [0.13154848, 0.27659297, 0.010708205, -0.11538539, 0.23766018]
  [1] dim=64, first 5: [0.23124999, 0.19641715, 0.22103494, -0.3130029, 0.11008452]
  [2] dim=64, first 5: [-0.0037580093, -0.17163365, 0.004440503, -0.07727925, 0.027430918]


()

## 4. Cosine similarity sanity check

Verify that semantically similar texts produce closer embeddings.

In [8]:
fn cosine_similarity(a: &[f32], b: &[f32]) -> f32 {
    let dot: f32 = a.iter().zip(b).map(|(x, y)| x * y).sum();
    let norm_a: f32 = a.iter().map(|x| x * x).sum::<f32>().sqrt();
    let norm_b: f32 = b.iter().map(|x| x * x).sum::<f32>().sqrt();
    dot / (norm_a * norm_b)
}

let query = model.encode_single("distributed systems conflict resolution");

for (i, text) in texts.iter().enumerate() {
    let sim = cosine_similarity(&query, &batch_embeddings[i]);
    println!("{:.4}  {}", sim, text);
}

println!("\nExpect: CRDT text closest, bread text furthest");

0.8299  CRDT conflict resolution in distributed systems
0.1754  Rust ownership model prevents data races
-0.1691  Making sourdough bread with a Dutch oven

Expect: CRDT text closest, bread text furthest


## 5. Model revision / identity

Can we get the model's dimension and revision SHA?

In [9]:
// Check if model2vec exposes dimension or revision info
// This will depend on the API — explore what's available
println!("Model type: {:?}", std::any::type_name_of_val(&model));

// Try to find the HF cache directory for revision
let home = std::env::var("HOME").unwrap();
let cache_path = format!("{}/.cache/huggingface/hub/models--minishlab--potion-base-2M/snapshots", home);
println!("\nLooking for cached snapshots at: {}", cache_path);

if let Ok(entries) = std::fs::read_dir(&cache_path) {
    for entry in entries {
        if let Ok(e) = entry {
            println!("  Snapshot: {}", e.file_name().to_string_lossy());
        }
    }
} else {
    println!("  Cache directory not found — model2vec-rs may use a different cache layout");
    
    // Check alternative cache locations
    let alt_path = format!("{}/.cache/mdvs/models", home);
    println!("  Checking {}", alt_path);
    if let Ok(entries) = std::fs::read_dir(&alt_path) {
        for entry in entries {
            if let Ok(e) = entry {
                println!("    {}", e.file_name().to_string_lossy());
            }
        }
    }
}

Model type: "model2vec_rs::model::StaticModel"

Looking for cached snapshots at: /Users/edoardo.chio/.cache/huggingface/hub/models--minishlab--potion-base-2M/snapshots
  Snapshot: 14a13cec05bc06a71787df81317c0de6b6d2fbf4


()

## 6. Long input / no context window

Static models (token lookup + average) have no transformer context window.
Verify that arbitrarily long input works, and check whether embedding quality
degrades as input grows (dilution from averaging many tokens).

In [10]:
{
    let base_sentence = "Conflict-free replicated data types enable distributed consensus. ";
    let base_embedding = model.encode_single(base_sentence);

    println!("=== Long input test: no context window ===\n");
    println!("{:<12} {:>8} {:>8}  sim vs base", "repetitions", "tokens†", "dim");
    println!("{}", "-".repeat(50));

    for reps in [1, 10, 100, 500, 1000] {
        let long_text = base_sentence.repeat(reps);
        let char_count = long_text.chars().count();

        let emb = model.encode_single(&long_text);
        let sim = cosine_similarity(&base_embedding, &emb);

        println!(
            "{:<12} {:>8} {:>8}  {:.4}",
            reps, char_count, emb.len(), sim
        );
    }

    println!("\n† approximate chars (static models tokenize internally)");
    println!("\nExpect: no crash at any length, similarity ~1.0 (same content repeated)");

    // Also test: long input with mixed content (dilution effect)
    println!("\n=== Dilution test: signal buried in noise ===\n");

    let signal = "CRDT conflict resolution in distributed systems. ";
    let noise = "Making sourdough bread with a Dutch oven. ";

    let signal_embedding = model.encode_single(signal);

    println!("{:<20} {:>6}  sim vs signal", "noise:signal ratio", "chars");
    println!("{}", "-".repeat(50));

    for noise_reps in [0, 1, 5, 10, 50] {
        let mixed = format!("{}{}", signal, noise.repeat(noise_reps));
        let emb = model.encode_single(&mixed);
        let sim = cosine_similarity(&signal_embedding, &emb);
        println!(
            "{}:1 {:<14} {:>6}  {:.4}",
            noise_reps,
            "",
            mixed.chars().count(),
            sim
        );
    }

    println!("\nExpect: similarity drops as noise overwhelms signal — justifies chunking");
}

=== Long input test: no context window ===

repetitions   tokens†      dim  sim vs base
--------------------------------------------------
1                  66       64  1.0000
10                660       64  1.0000
100              6600       64  1.0000
500             33000       64  1.0000
1000            66000       64  1.0000

† approximate chars (static models tokenize internally)

Expect: no crash at any length, similarity ~1.0 (same content repeated)

=== Dilution test: signal buried in noise ===

noise:signal ratio    chars  sim vs signal
--------------------------------------------------
0:1                    49  1.0000
1:1                    91  0.5039
5:1                   259  0.0363
10:1                   469  -0.0321
50:1                  2149  -0.0865

Expect: similarity drops as noise overwhelms signal — justifies chunking


()

## Findings

- [x] API: `StaticModel::from_pretrained(repo, token, normalize, subfolder)` — 4 args, not 3
- [x] Output dimension for potion-base-2M: **64** (not 256 as README assumed)
- [x] Batch: `model.encode(&[String])` → `Vec<Vec<f32>>` — needs `String` not `&str`
- [x] Single: `model.encode_single(&str)` → `Vec<f32>`
- [x] Revision resolution: **HF cache works** — `~/.cache/huggingface/hub/models--{org}--{name}/snapshots/{sha}/`
- [x] `StaticModel` is not exposed by API, must read from filesystem
- [x] Embedding quality: clear separation — CRDT 0.83, Rust 0.18, bread -0.17 for a distributed systems query
- [x] Cosine similarity ranges -1 to +1 (DuckDB `array_cosine_distance` returns 1-similarity, range 0 to 2)
- [x] **No context window** — 66k chars (1000x repeat) works with no crash, perfect similarity. Static models have no input length limit.
- [x] **Dilution justifies chunking** — at 5:1 noise:signal ratio similarity drops to 0.04, at 10:1 goes negative (-0.03). Chunking is about semantic quality, not model limits.

**Important**: README listed potion-base-2M as 256-dim — it's actually 64. Need to verify dimensions for all POTION models before choosing a default.